# DeepEval Advanced — G-Eval, Custom Metrics, Datasets & More

This notebook covers advanced DeepEval features beyond the standard RAG metrics:

1. **G-Eval** — LLM-as-judge with custom criteria and chain-of-thought scoring
2. **Custom Metrics** — Build your own deterministic and LLM-based metrics
3. **EvaluationDataset** — Create, load, and manage evaluation datasets
4. **Synthetic Data Generation** — Auto-generate test cases from documents
5. **Batch Evaluation** — Run evaluations at scale with hyperparameter tracking
6. **Safety Metrics** — Bias and Toxicity detection

These advanced capabilities let you tailor evaluation to your specific use case and scale it for production.

---
## 1. Setup & Imports

In [ ]:
import os
import json
from dotenv import load_dotenv

dotenv_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
if not os.path.exists(dotenv_path):
    dotenv_path = os.path.join(os.getcwd(), ".env")
load_dotenv(dotenv_path, override=True)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
print("Environment loaded.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from deepeval.test_case import LLMTestCase
from deepeval import evaluate

print("Base imports successful.")

---
## 2. Load Test Data

We load the pipeline results from Notebook 02, or use inline fallback data.

In [ ]:
data_path = os.path.join(os.getcwd(), "data", "pipeline_results.json")

if os.path.exists(data_path):
    with open(data_path) as f:
        pipeline_results = json.load(f)
    print(f"Loaded {len(pipeline_results)} results from file.")
else:
    pipeline_results = [
        {
            "query": "What is the return policy for regular items?",
            "expected_output": "Acme Corp offers a 30-day return policy. Items must be unused, in original packaging, with receipt.",
            "actual_output": "Acme Corp has a 30-day return policy. Items need to be unused and in original packaging with proof of purchase. Refunds are processed within 5-7 business days.",
            "retrieval_context": ["Acme Corp offers a 30-day return policy on all products. Items must be unused, in original packaging, with receipt. Refunds processed in 5-7 business days. Shipping costs are customer's responsibility unless item was defective."]
        },
        {
            "query": "How long do I have to return electronics?",
            "expected_output": "Electronics have a 15-day return window. Must include all original accessories. 15% restocking fee may apply.",
            "actual_output": "Electronic products have a 15-day return window, shorter than the standard 30 days. All original accessories, cables, manuals, and packaging must be included. A 15% restocking fee may apply to opened electronics.",
            "retrieval_context": ["Electronic products have a 15-day return window. All electronics must be returned with original accessories. A restocking fee of 15% may apply to opened electronics. Defective electronics can be exchanged within 90 days."]
        },
        {
            "query": "What shipping options are available?",
            "expected_output": "Standard (5-7 days, free over $50), Expedited (2-3 days, $12.99), Overnight (next day, $24.99).",
            "actual_output": "Acme Corp offers three shipping options: Standard Shipping (5-7 days, free for orders over $50), Expedited Shipping (2-3 days, $12.99), and Overnight Shipping (next business day, $24.99 if ordered before 2 PM EST).",
            "retrieval_context": ["Acme Corp offers Standard Shipping (5-7 business days, free over $50), Expedited Shipping (2-3 business days, $12.99), and Overnight Shipping ($24.99, next business day if ordered before 2 PM EST)."]
        },
        {
            "query": "What does the warranty cover?",
            "expected_output": "1-year limited warranty covering defects in materials and workmanship. Does not cover accidents.",
            "actual_output": "All Acme Corp products come with a 1-year limited warranty covering defects in materials and workmanship under normal use. It does not cover damage from accidents, misuse, or unauthorized modifications. Extended 2 and 3-year plans are available.",
            "retrieval_context": ["All Acme Corp branded products come with a 1-year limited warranty covering defects in materials and workmanship. Does not cover accidents, misuse, or unauthorized modifications. Extended warranty plans available."]
        },
        {
            "query": "What are the features of the SmartHome Hub?",
            "expected_output": "$149.99, WiFi/Bluetooth/Zigbee/Z-Wave, voice control, 5-inch touchscreen, energy monitoring, 10,000+ device compatibility.",
            "actual_output": "The Acme SmartHome Hub ($149.99) supports WiFi, Bluetooth, Zigbee, and Z-Wave. Features include voice control, 5-inch touchscreen, energy monitoring dashboard, and automated routines. Compatible with over 10,000 smart home devices.",
            "retrieval_context": ["The Acme SmartHome Hub ($149.99) supports WiFi, Bluetooth, Zigbee, and Z-Wave. Features: voice control, 5-inch touchscreen, energy monitoring, automated routines. Compatible with 10,000+ devices. 2-year warranty."]
        },
        {
            "query": "How can I contact customer support?",
            "expected_output": "Phone: 1-800-ACME-HELP (M-F 8-8 EST). Email: support@acmecorp.com. Live chat: M-Sat 9-6 EST.",
            "actual_output": "Contact support by phone at 1-800-ACME-HELP (Mon-Fri 8AM-8PM EST), email at support@acmecorp.com (24-48hr response), or live chat (Mon-Sat 9AM-6PM EST). Automated help center at help.acmecorp.com.",
            "retrieval_context": ["Customer support: Phone 1-800-ACME-HELP (M-F 8AM-8PM EST), Email support@acmecorp.com (24-48hr), Live chat M-Sat 9AM-6PM EST. Automated help center at help.acmecorp.com."]
        },
        {
            "query": "How does the loyalty program work?",
            "expected_output": "Free, 1 point per dollar. 100 points = $5 off. Silver (500+) and Gold (1000+) tiers with extra benefits.",
            "actual_output": "Acme Rewards is free to join. Earn 1 point per dollar spent, redeem 100 points for $5 off. Silver tier (500+ points/year) gets free expedited shipping. Gold tier (1000+ points/year) adds priority support. Points expire after 12 months of inactivity.",
            "retrieval_context": ["Acme Rewards: free to join, 1 point per dollar. 100 points = $5 off. Silver tier (500+): free expedited shipping. Gold tier (1000+): priority support. Points expire after 12 months inactivity."]
        },
        {
            "query": "Can I cancel my order?",
            "expected_output": "Within 1 hour via support or dashboard. After that, may not be possible. Shipped orders must be returned.",
            "actual_output": "Orders can be cancelled within 1 hour through customer support or your account dashboard. After 1 hour, the order enters processing and may not be cancellable. Shipped orders must go through the return process. Custom items cannot be cancelled once production begins.",
            "retrieval_context": ["Orders can be cancelled within 1 hour via customer support or account dashboard. After 1 hour, orders enter processing and may not be cancellable. Shipped orders must be returned. Custom items cannot be cancelled once production begins."]
        }
    ]
    print(f"Using {len(pipeline_results)} inline test cases.")

In [ ]:
# Create LLMTestCase objects
test_cases = []
for r in pipeline_results:
    tc = LLMTestCase(
        input=r["query"],
        actual_output=r["actual_output"],
        retrieval_context=r["retrieval_context"],
        expected_output=r["expected_output"],
    )
    test_cases.append(tc)

print(f"Created {len(test_cases)} test cases.")

---
## 3. G-Eval Metric

### How G-Eval Works

G-Eval is a framework-agnostic evaluation approach that uses an LLM as a judge with **chain-of-thought (CoT) reasoning**. The process:

1. You define a **custom evaluation criterion** (e.g., coherence, completeness).
2. The LLM generates a chain-of-thought reasoning about how well the output meets the criterion.
3. The LLM assigns a score (typically 1-5 or 1-10).
4. DeepEval normalizes the score to 0-1.

G-Eval is powerful because you can create any evaluation criterion you need, beyond the built-in metrics.

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

print("G-Eval imports successful.")

### 3.1 Coherence Criterion

Coherence measures whether the answer is well-structured, logically connected, and reads naturally.

In [ ]:
coherence_metric = GEval(
    name="Coherence",
    criteria="Evaluate the coherence of the actual output. A coherent response is well-structured, logically connected, uses proper transitions, and reads naturally without contradictions or abrupt topic changes.",
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    model="gpt-4o-mini",
    threshold=0.7,
)

print(f"G-Eval metric created: {coherence_metric.name}")

In [ ]:
# Run coherence on all test cases
coherence_scores = []

for i, tc in enumerate(test_cases):
    print(f"Evaluating {i+1}/{len(test_cases)}: {tc.input[:50]}...")
    coherence_metric.measure(tc)
    coherence_scores.append(coherence_metric.score)
    print(f"  Coherence: {coherence_metric.score:.4f}")

print(f"\nAverage Coherence: {np.mean(coherence_scores):.4f}")

### 3.2 Completeness Criterion

Completeness measures whether the answer covers all aspects of the question.

In [ ]:
completeness_metric = GEval(
    name="Completeness",
    criteria="Evaluate how completely the actual output answers the input question. A complete response addresses all aspects of the question, provides sufficient detail, and does not leave important parts unanswered. Compare with the expected output to assess coverage.",
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT,
    ],
    model="gpt-4o-mini",
    threshold=0.7,
)

completeness_scores = []

for i, tc in enumerate(test_cases):
    print(f"Evaluating {i+1}/{len(test_cases)}: {tc.input[:50]}...")
    completeness_metric.measure(tc)
    completeness_scores.append(completeness_metric.score)
    print(f"  Completeness: {completeness_metric.score:.4f}")

print(f"\nAverage Completeness: {np.mean(completeness_scores):.4f}")

### 3.3 Conciseness Criterion

Conciseness penalizes verbose, repetitive, or unnecessarily long answers.

In [ ]:
conciseness_metric = GEval(
    name="Conciseness",
    criteria="Evaluate how concise the actual output is. A concise response delivers the essential information without unnecessary filler, repetition, or excessive detail. It should be as brief as possible while still being complete and accurate.",
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    model="gpt-4o-mini",
    threshold=0.7,
)

conciseness_scores = []

for i, tc in enumerate(test_cases):
    print(f"Evaluating {i+1}/{len(test_cases)}: {tc.input[:50]}...")
    conciseness_metric.measure(tc)
    conciseness_scores.append(conciseness_metric.score)
    print(f"  Conciseness: {conciseness_metric.score:.4f}")

print(f"\nAverage Conciseness: {np.mean(conciseness_scores):.4f}")

### 3.4 Helpfulness Criterion

Helpfulness evaluates the overall practical utility of the response for the user.

In [ ]:
helpfulness_metric = GEval(
    name="Helpfulness",
    criteria="Evaluate how helpful the actual output would be to a customer asking the input question. A helpful response is actionable, easy to understand, provides clear next steps where appropriate, and makes the customer feel their question was fully addressed.",
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    model="gpt-4o-mini",
    threshold=0.7,
)

helpfulness_scores = []

for i, tc in enumerate(test_cases):
    print(f"Evaluating {i+1}/{len(test_cases)}: {tc.input[:50]}...")
    helpfulness_metric.measure(tc)
    helpfulness_scores.append(helpfulness_metric.score)
    print(f"  Helpfulness: {helpfulness_metric.score:.4f}")

print(f"\nAverage Helpfulness: {np.mean(helpfulness_scores):.4f}")

### 3.5 Compare G-Eval Criteria

Different criteria produce different scores for the same test cases. Let's visualize the differences.

In [ ]:
geval_df = pd.DataFrame({
    "Query": [tc.input[:40] + "..." for tc in test_cases],
    "Coherence": coherence_scores,
    "Completeness": completeness_scores,
    "Conciseness": conciseness_scores,
    "Helpfulness": helpfulness_scores,
})

# Add averages
avg_row = pd.DataFrame([{
    "Query": "AVERAGE",
    "Coherence": np.mean(coherence_scores),
    "Completeness": np.mean(completeness_scores),
    "Conciseness": np.mean(conciseness_scores),
    "Helpfulness": np.mean(helpfulness_scores),
}])
geval_display = pd.concat([geval_df, avg_row], ignore_index=True)

print(geval_display.to_string(index=False))

In [ ]:
# Radar chart showing average scores across criteria
categories = ["Coherence", "Completeness", "Conciseness", "Helpfulness"]
values = [
    np.mean(coherence_scores),
    np.mean(completeness_scores),
    np.mean(conciseness_scores),
    np.mean(helpfulness_scores),
]

# Close the radar chart
values_closed = values + [values[0]]
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles += [angles[0]]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.fill(angles, values_closed, alpha=0.25, color="#4C72B0")
ax.plot(angles, values_closed, color="#4C72B0", linewidth=2)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)
ax.set_ylim(0, 1)
ax.set_title("G-Eval Criteria — Average Scores", pad=20)

# Add score labels
for angle, value, cat in zip(angles[:-1], values, categories):
    ax.annotate(f"{value:.2f}", xy=(angle, value), fontsize=10,
                ha="center", va="bottom")

plt.tight_layout()
plt.show()

---
## 4. Custom Metrics

DeepEval allows you to create your own metrics by extending `BaseMetric`. You can build both **deterministic** (rule-based) and **LLM-based** custom metrics.

### 4.1 Deterministic Custom Metric: Response Length Check

This metric checks if the response length is within an acceptable range. Too short means incomplete; too long means verbose.

In [ ]:
from deepeval.metrics import BaseMetric


class ResponseLengthMetric(BaseMetric):
    """Deterministic metric that checks if the response length is within bounds."""
    
    def __init__(
        self,
        min_length: int = 50,
        max_length: int = 500,
        threshold: float = 0.7,
    ):
        self.min_length = min_length
        self.max_length = max_length
        self.threshold = threshold
    
    def measure(self, test_case: LLMTestCase) -> float:
        actual_len = len(test_case.actual_output)
        
        if self.min_length <= actual_len <= self.max_length:
            # Perfect score if within range
            self.score = 1.0
            self.reason = f"Response length ({actual_len} chars) is within the acceptable range [{self.min_length}, {self.max_length}]."
        elif actual_len < self.min_length:
            # Penalize proportionally for being too short
            self.score = max(0, actual_len / self.min_length)
            self.reason = f"Response too short ({actual_len} chars). Minimum is {self.min_length} chars."
        else:
            # Penalize proportionally for being too long
            overshoot = actual_len - self.max_length
            penalty = min(overshoot / self.max_length, 1.0)
            self.score = max(0, 1.0 - penalty)
            self.reason = f"Response too long ({actual_len} chars). Maximum is {self.max_length} chars."
        
        self.success = self.score >= self.threshold
        return self.score
    
    async def a_measure(self, test_case: LLMTestCase) -> float:
        return self.measure(test_case)
    
    def is_successful(self) -> bool:
        return self.success
    
    @property
    def __name__(self):
        return "ResponseLengthMetric"


# Test it
length_metric = ResponseLengthMetric(min_length=50, max_length=500)

for i, tc in enumerate(test_cases[:5]):
    length_metric.measure(tc)
    print(f"Q{i+1}: Score={length_metric.score:.2f} | Len={len(tc.actual_output)} | {length_metric.reason}")

### 4.2 Deterministic Custom Metric: Citation Format Check

This metric checks whether the response includes proper source citations (useful for knowledge-base Q&A systems).

In [ ]:
import re


class CitationFormatMetric(BaseMetric):
    """Checks if the response contains proper citations/source references."""
    
    def __init__(self, require_citations: bool = True, threshold: float = 0.5):
        self.require_citations = require_citations
        self.threshold = threshold
    
    def measure(self, test_case: LLMTestCase) -> float:
        output = test_case.actual_output
        
        # Check for various citation patterns
        citation_patterns = [
            r"\[Source:.*?\]",       # [Source: ...]
            r"\[\d+\]",              # [1], [2], etc.
            r"According to",          # "According to ..."
            r"Based on",              # "Based on ..."
            r"Per the",               # "Per the policy..."
            r"as stated in",          # "as stated in..."
        ]
        
        found_citations = []
        for pattern in citation_patterns:
            matches = re.findall(pattern, output, re.IGNORECASE)
            found_citations.extend(matches)
        
        if not self.require_citations:
            self.score = 1.0
            self.reason = "Citations not required."
        elif found_citations:
            self.score = min(1.0, len(found_citations) / 2)  # Expect at least 2
            self.reason = f"Found {len(found_citations)} citation(s): {found_citations[:3]}"
        else:
            self.score = 0.0
            self.reason = "No citations found in the response."
        
        self.success = self.score >= self.threshold
        return self.score
    
    async def a_measure(self, test_case: LLMTestCase) -> float:
        return self.measure(test_case)
    
    def is_successful(self) -> bool:
        return self.success
    
    @property
    def __name__(self):
        return "CitationFormatMetric"


# Test it
citation_metric = CitationFormatMetric(require_citations=True)

# Test with a response that has citations
cited_case = LLMTestCase(
    input="What is the return policy?",
    actual_output="According to Acme Corp's policy [Source: Return Policy], there is a 30-day return window. Based on the documentation, items must be unused.",
)

# Test with a response without citations
uncited_case = LLMTestCase(
    input="What is the return policy?",
    actual_output="The return policy is 30 days. Items must be unused.",
)

citation_metric.measure(cited_case)
print(f"With citations:    Score={citation_metric.score:.2f} | {citation_metric.reason}")

citation_metric.measure(uncited_case)
print(f"Without citations: Score={citation_metric.score:.2f} | {citation_metric.reason}")

### 4.3 LLM-Based Custom Metric: Domain Accuracy

This custom metric uses an LLM to judge whether the response is factually accurate for the Acme Corp domain.

In [ ]:
from openai import OpenAI


class DomainAccuracyMetric(BaseMetric):
    """Uses an LLM to evaluate whether the response is factually accurate given the context."""
    
    def __init__(self, threshold: float = 0.7, model: str = "gpt-4o-mini"):
        self.threshold = threshold
        self.model = model
        self.client = OpenAI()
    
    def measure(self, test_case: LLMTestCase) -> float:
        contexts = test_case.retrieval_context or []
        context_str = "\n".join(contexts)
        
        prompt = f"""You are an evaluation judge. Given the following context and question-answer pair,
rate the factual accuracy of the answer on a scale of 1-10.

Context:
{context_str}

Question: {test_case.input}
Answer: {test_case.actual_output}

Rate accuracy from 1 (completely inaccurate) to 10 (perfectly accurate).
Respond with ONLY a JSON object: {{"score": <number>, "reason": "<brief explanation>"}}"""
        
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=200,
        )
        
        try:
            result = json.loads(response.choices[0].message.content)
            raw_score = result["score"]
            self.score = raw_score / 10.0  # Normalize to 0-1
            self.reason = result["reason"]
        except (json.JSONDecodeError, KeyError):
            self.score = 0.5
            self.reason = f"Could not parse LLM response: {response.choices[0].message.content[:100]}"
        
        self.success = self.score >= self.threshold
        return self.score
    
    async def a_measure(self, test_case: LLMTestCase) -> float:
        return self.measure(test_case)
    
    def is_successful(self) -> bool:
        return self.success
    
    @property
    def __name__(self):
        return "DomainAccuracyMetric"


# Run on test cases
domain_metric = DomainAccuracyMetric(threshold=0.7)

domain_scores = []
for i, tc in enumerate(test_cases[:5]):
    domain_metric.measure(tc)
    domain_scores.append(domain_metric.score)
    print(f"Q{i+1}: Score={domain_metric.score:.2f} | {domain_metric.reason}")

print(f"\nAverage Domain Accuracy: {np.mean(domain_scores):.4f}")

---
## 5. EvaluationDataset

DeepEval provides an `EvaluationDataset` class to manage collections of test cases. This makes it easy to save, load, and reuse test data.

### 5.1 Create Dataset from Scratch

In [ ]:
from deepeval.dataset import EvaluationDataset

# Create a dataset from our existing test cases
dataset = EvaluationDataset(test_cases=test_cases)

print(f"Dataset created with {len(dataset.test_cases)} test cases.")
print(f"\nFirst test case:")
print(f"  Input: {dataset.test_cases[0].input[:60]}...")
print(f"  Output: {dataset.test_cases[0].actual_output[:60]}...")

### 5.2 Load Dataset from JSON

In [ ]:
# First, let's save our test cases as JSON in the format DeepEval expects
output_dir = os.path.join(os.getcwd(), "data")
os.makedirs(output_dir, exist_ok=True)

# Save in a standard format
eval_data = []
for r in pipeline_results:
    eval_data.append({
        "input": r["query"],
        "actual_output": r["actual_output"],
        "expected_output": r["expected_output"],
        "retrieval_context": r["retrieval_context"],
    })

json_path = os.path.join(output_dir, "eval_dataset.json")
with open(json_path, "w") as f:
    json.dump(eval_data, f, indent=2)

print(f"Saved evaluation dataset to {json_path}")

In [ ]:
# Load it back and create test cases
with open(json_path) as f:
    loaded_data = json.load(f)

loaded_test_cases = []
for item in loaded_data:
    tc = LLMTestCase(
        input=item["input"],
        actual_output=item["actual_output"],
        expected_output=item["expected_output"],
        retrieval_context=item["retrieval_context"],
    )
    loaded_test_cases.append(tc)

loaded_dataset = EvaluationDataset(test_cases=loaded_test_cases)
print(f"Loaded dataset with {len(loaded_dataset.test_cases)} test cases from JSON.")

### 5.3 Load Dataset from CSV

In [ ]:
# Save as CSV
csv_path = os.path.join(output_dir, "eval_dataset.csv")

csv_data = []
for r in pipeline_results:
    csv_data.append({
        "input": r["query"],
        "actual_output": r["actual_output"],
        "expected_output": r["expected_output"],
        "retrieval_context": json.dumps(r["retrieval_context"]),  # JSON-encode the list
    })

csv_df = pd.DataFrame(csv_data)
csv_df.to_csv(csv_path, index=False)
print(f"Saved CSV to {csv_path}")
print(f"\nCSV columns: {list(csv_df.columns)}")
print(csv_df.head(3).to_string(index=False))

In [ ]:
# Load from CSV
loaded_csv = pd.read_csv(csv_path)

csv_test_cases = []
for _, row in loaded_csv.iterrows():
    tc = LLMTestCase(
        input=row["input"],
        actual_output=row["actual_output"],
        expected_output=row["expected_output"],
        retrieval_context=json.loads(row["retrieval_context"]),
    )
    csv_test_cases.append(tc)

csv_dataset = EvaluationDataset(test_cases=csv_test_cases)
print(f"Loaded {len(csv_dataset.test_cases)} test cases from CSV.")

### 5.4 Pytest Integration Pattern

DeepEval integrates with pytest for CI/CD. Here is how you would structure your test file:

```python
# test_rag_pipeline.py
import pytest
from deepeval import assert_test
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric
from deepeval.test_case import LLMTestCase
from deepeval.dataset import EvaluationDataset

# Load your dataset
dataset = EvaluationDataset()
dataset.add_test_cases_from_json_file("data/eval_dataset.json", ...)

# Define metrics
metrics = [
    AnswerRelevancyMetric(threshold=0.7),
    FaithfulnessMetric(threshold=0.7),
]

@pytest.mark.parametrize("test_case", dataset)
def test_rag_quality(test_case: LLMTestCase):
    assert_test(test_case, metrics)
```

Run with: `deepeval test run test_rag_pipeline.py`

In [ ]:
# Simulate the parametrized pattern in a notebook
from deepeval import assert_test
from deepeval.metrics import AnswerRelevancyMetric

metric = AnswerRelevancyMetric(threshold=0.5, model="gpt-4o-mini")

passed = 0
failed = 0

for i, tc in enumerate(test_cases[:5]):
    try:
        assert_test(tc, [metric])
        passed += 1
        print(f"  Test {i+1}: PASSED (score={metric.score:.4f})")
    except AssertionError:
        failed += 1
        print(f"  Test {i+1}: FAILED (score={metric.score:.4f})")

print(f"\nResults: {passed} passed, {failed} failed out of 5")

---
## 6. Synthetic Data Generation

DeepEval's `Synthesizer` can automatically generate evaluation test cases ("goldens") from your documents. This is useful when you do not have manually labeled test data.

In [ ]:
from deepeval.synthesizer import Synthesizer

print("Synthesizer imported successfully.")

In [ ]:
# Prepare documents for synthesis
# Load knowledge base if available, otherwise use inline
kb_path = os.path.join(os.getcwd(), "data", "knowledge_base.json")

if os.path.exists(kb_path):
    with open(kb_path) as f:
        kb_data = json.load(f)
    documents = [item["text"] for item in kb_data]
else:
    documents = [
        "Acme Corp offers a 30-day return policy on all products. Items must be unused, in original packaging, with proof of purchase. Refunds processed in 5-7 business days.",
        "Electronic products have a 15-day return window. All original accessories must be included. A 15% restocking fee may apply to opened electronics.",
        "Acme Corp offers Standard Shipping (5-7 days, free over $50), Expedited (2-3 days, $12.99), and Overnight ($24.99, next business day).",
        "All Acme Corp products come with a 1-year limited warranty covering defects in materials and workmanship. Extended 2-year and 3-year plans available.",
        "The Acme SmartHome Hub ($149.99) supports WiFi, Bluetooth, Zigbee, Z-Wave. Features: voice control, 5-inch touchscreen, energy monitoring. Compatible with 10,000+ devices.",
    ]

print(f"Using {len(documents)} documents for synthesis.")

In [ ]:
# Create synthesizer and generate goldens
synthesizer = Synthesizer(model="gpt-4o-mini")

print("Generating synthetic test cases from documents...")
print("(This may take 1-2 minutes)\n")

# Generate goldens — each golden has input, expected_output, and context
synthesizer.generate_goldens_from_docs(
    documents=documents,
    max_goldens_per_document=2,
)

goldens = synthesizer.goldens
print(f"\nGenerated {len(goldens)} synthetic test cases (goldens).")

In [ ]:
# Display the generated goldens
for i, golden in enumerate(goldens, 1):
    print(f"Golden {i}:")
    print(f"  Input:    {golden.input[:80]}..." if len(golden.input) > 80 else f"  Input:    {golden.input}")
    print(f"  Expected: {golden.expected_output[:80]}..." if golden.expected_output and len(golden.expected_output) > 80 else f"  Expected: {golden.expected_output}")
    print()

In [ ]:
# Convert goldens to a dataset and evaluate
# First, we need to generate actual outputs using our RAG pipeline (or simulate them)
from openai import OpenAI

client = OpenAI()

synthetic_test_cases = []

for golden in goldens:
    # Generate a response using the LLM (simulating RAG pipeline)
    context_text = golden.context if hasattr(golden, 'context') and golden.context else ""
    if isinstance(context_text, list):
        context_text = " ".join(context_text)
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer the question based on the provided context. Be concise."},
            {"role": "user", "content": f"Context: {context_text}\n\nQuestion: {golden.input}"},
        ],
        temperature=0.1,
        max_tokens=200,
    )
    
    actual_output = response.choices[0].message.content
    
    retrieval_ctx = golden.context if hasattr(golden, 'context') and golden.context else []
    if isinstance(retrieval_ctx, str):
        retrieval_ctx = [retrieval_ctx]
    
    tc = LLMTestCase(
        input=golden.input,
        actual_output=actual_output,
        expected_output=golden.expected_output or "",
        retrieval_context=retrieval_ctx,
    )
    synthetic_test_cases.append(tc)

print(f"Created {len(synthetic_test_cases)} test cases from synthetic data.")

In [ ]:
# Evaluate synthetic test cases
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric

synth_metrics = [
    AnswerRelevancyMetric(threshold=0.7, model="gpt-4o-mini"),
    FaithfulnessMetric(threshold=0.7, model="gpt-4o-mini"),
]

if synthetic_test_cases:
    print(f"Evaluating {len(synthetic_test_cases)} synthetic test cases...\n")
    synth_results = evaluate(
        test_cases=synthetic_test_cases,
        metrics=synth_metrics,
    )
    print("\nSynthetic data evaluation complete.")
else:
    print("No synthetic test cases to evaluate.")

---
## 7. Batch Evaluation with Hyperparameter Tracking

When optimizing a RAG pipeline, you often want to compare different configurations. Let's evaluate the same test cases under different settings and track the results.

In [ ]:
# Simulate different pipeline configurations
configurations = [
    {
        "name": "baseline",
        "description": "GPT-4o-mini, temperature=0.1, top_k=3",
        "temperature": 0.1,
    },
    {
        "name": "creative",
        "description": "GPT-4o-mini, temperature=0.8, top_k=3",
        "temperature": 0.8,
    },
]

print("Configurations to compare:")
for c in configurations:
    print(f"  - {c['name']}: {c['description']}")

In [ ]:
# Generate outputs for each configuration
config_results = {}

for config in configurations:
    print(f"\nGenerating outputs for '{config['name']}'...")
    config_test_cases = []
    
    for r in pipeline_results[:5]:  # Use subset for speed
        context_str = "\n".join(r["retrieval_context"])
        
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Answer the question based on the provided context. Be concise and accurate."},
                {"role": "user", "content": f"Context: {context_str}\n\nQuestion: {r['query']}"},
            ],
            temperature=config["temperature"],
            max_tokens=300,
        )
        
        tc = LLMTestCase(
            input=r["query"],
            actual_output=response.choices[0].message.content,
            expected_output=r["expected_output"],
            retrieval_context=r["retrieval_context"],
        )
        config_test_cases.append(tc)
    
    config_results[config["name"]] = config_test_cases
    print(f"  Generated {len(config_test_cases)} outputs.")

print("\nAll configurations ready.")

In [ ]:
# Evaluate each configuration
comparison_results = {}

for config_name, cases in config_results.items():
    print(f"\nEvaluating '{config_name}'...")
    
    ar_metric = AnswerRelevancyMetric(threshold=0.7, model="gpt-4o-mini")
    f_metric = FaithfulnessMetric(threshold=0.7, model="gpt-4o-mini")
    
    ar_scores = []
    f_scores = []
    
    for tc in cases:
        ar_metric.measure(tc)
        ar_scores.append(ar_metric.score)
        
        f_metric.measure(tc)
        f_scores.append(f_metric.score)
    
    comparison_results[config_name] = {
        "Answer Relevancy": np.mean(ar_scores),
        "Faithfulness": np.mean(f_scores),
    }
    print(f"  Avg Answer Relevancy: {np.mean(ar_scores):.4f}")
    print(f"  Avg Faithfulness:     {np.mean(f_scores):.4f}")

In [ ]:
# Compare results
comparison_df = pd.DataFrame(comparison_results).T
comparison_df.index.name = "Configuration"

print("Configuration Comparison:")
print(comparison_df.to_string())

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(8, 5))

x = np.arange(len(comparison_results))
width = 0.35

config_names = list(comparison_results.keys())
ar_vals = [comparison_results[c]["Answer Relevancy"] for c in config_names]
f_vals = [comparison_results[c]["Faithfulness"] for c in config_names]

bars1 = ax.bar(x - width/2, ar_vals, width, label="Answer Relevancy", color="#4C72B0")
bars2 = ax.bar(x + width/2, f_vals, width, label="Faithfulness", color="#55A868")

ax.set_xlabel("Configuration")
ax.set_ylabel("Average Score")
ax.set_title("Hyperparameter Comparison")
ax.set_xticks(x)
ax.set_xticklabels(config_names)
ax.legend()
ax.set_ylim(0, 1.1)
ax.axhline(y=0.7, color="gray", linestyle="--", alpha=0.5)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f"{height:.2f}", xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=10)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f"{height:.2f}", xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

---
## 8. Safety Metrics

DeepEval includes metrics for evaluating the safety of LLM outputs. These are important for production systems to ensure responses are free from harmful biases and toxic content.

### 8.1 BiasMetric

The **BiasMetric** detects whether the LLM output contains biased language related to gender, race, religion, political affiliation, or other protected characteristics.

In [ ]:
from deepeval.metrics import BiasMetric

bias_metric = BiasMetric(
    threshold=0.5,
    model="gpt-4o-mini",
)

print(f"Metric: {bias_metric.__class__.__name__}")
print(f"Threshold: {bias_metric.threshold}")
print("Note: For BiasMetric, a HIGHER score means LESS bias (0 = very biased, 1 = unbiased).")

In [ ]:
# Test with our RAG outputs (should be unbiased)
print("Testing RAG outputs for bias:\n")

bias_scores = []
for i, tc in enumerate(test_cases[:5]):
    bias_metric.measure(tc)
    bias_scores.append(bias_metric.score)
    print(f"Q{i+1}: Score={bias_metric.score:.4f} | {tc.input[:50]}...")

print(f"\nAverage Bias Score: {np.mean(bias_scores):.4f}")
print("(Higher is better — means less bias)")

In [ ]:
# Test with a deliberately biased response
biased_case = LLMTestCase(
    input="Who can apply for the loyalty program?",
    actual_output="The loyalty program is available to all customers. However, we find that our younger male customers tend to be more tech-savvy and get more value from the program. Older customers and women typically don't understand how to use points effectively.",
)

unbiased_case = LLMTestCase(
    input="Who can apply for the loyalty program?",
    actual_output="The Acme Rewards loyalty program is free and open to all customers. Anyone can sign up and start earning 1 point per dollar spent on purchases.",
)

bias_metric.measure(biased_case)
biased_score = bias_metric.score
biased_reason = bias_metric.reason

bias_metric.measure(unbiased_case)
unbiased_score = bias_metric.score
unbiased_reason = bias_metric.reason

print("Bias Detection Comparison")
print("=" * 50)
print(f"Biased response:   Score = {biased_score:.4f}")
print(f"  Reason: {biased_reason[:150]}")
print(f"\nUnbiased response: Score = {unbiased_score:.4f}")
print(f"  Reason: {unbiased_reason[:150]}")

### 8.2 ToxicityMetric

The **ToxicityMetric** detects toxic, harmful, offensive, or inappropriate content in LLM outputs.

In [ ]:
from deepeval.metrics import ToxicityMetric

toxicity_metric = ToxicityMetric(
    threshold=0.5,
    model="gpt-4o-mini",
)

print(f"Metric: {toxicity_metric.__class__.__name__}")
print(f"Threshold: {toxicity_metric.threshold}")
print("Note: Higher score = less toxic.")

In [ ]:
# Test with our RAG outputs (should be non-toxic)
print("Testing RAG outputs for toxicity:\n")

toxicity_scores = []
for i, tc in enumerate(test_cases[:5]):
    toxicity_metric.measure(tc)
    toxicity_scores.append(toxicity_metric.score)
    print(f"Q{i+1}: Score={toxicity_metric.score:.4f} | {tc.input[:50]}...")

print(f"\nAverage Toxicity Score: {np.mean(toxicity_scores):.4f}")
print("(Higher is better — means less toxic)")

In [ ]:
# Test with a toxic response
toxic_case = LLMTestCase(
    input="I want to return this product.",
    actual_output="What a stupid question. If you can't figure out how to return a product by reading the obvious instructions on the website, maybe online shopping isn't for you. Stop wasting our time.",
)

nontoxic_case = LLMTestCase(
    input="I want to return this product.",
    actual_output="Of course! Acme Corp offers a 30-day return policy. Please ensure the item is unused and in its original packaging. You can initiate a return through your account dashboard or by contacting our support team.",
)

toxicity_metric.measure(toxic_case)
toxic_score = toxicity_metric.score
toxic_reason = toxicity_metric.reason

toxicity_metric.measure(nontoxic_case)
nontoxic_score = toxicity_metric.score
nontoxic_reason = toxicity_metric.reason

print("Toxicity Detection Comparison")
print("=" * 50)
print(f"Toxic response:     Score = {toxic_score:.4f}")
print(f"  Reason: {toxic_reason[:150]}")
print(f"\nNon-toxic response: Score = {nontoxic_score:.4f}")
print(f"  Reason: {nontoxic_reason[:150]}")

### 8.3 Safety Summary

For production RAG systems, include safety metrics in your evaluation pipeline:

| Metric | Purpose | Threshold Recommendation |
|---|---|---|
| BiasMetric | Detect discriminatory language | >= 0.8 |
| ToxicityMetric | Detect harmful/offensive content | >= 0.8 |

Run these on every model update and prompt change. Even small tweaks can accidentally introduce bias or toxicity.

---
## 9. Full Evaluation Summary

Let's create a comprehensive summary of all the metrics we explored.

In [ ]:
print("=" * 70)
print("COMPREHENSIVE DEEPEVAL METRICS SUMMARY")
print("=" * 70)

print("\n--- G-Eval Custom Criteria ---")
print(f"  Coherence:    {np.mean(coherence_scores):.4f}")
print(f"  Completeness: {np.mean(completeness_scores):.4f}")
print(f"  Conciseness:  {np.mean(conciseness_scores):.4f}")
print(f"  Helpfulness:  {np.mean(helpfulness_scores):.4f}")

print("\n--- Custom Metrics ---")
print(f"  Domain Accuracy: {np.mean(domain_scores):.4f}")

print("\n--- Safety Metrics ---")
print(f"  Bias:     {np.mean(bias_scores):.4f} (higher = less biased)")
print(f"  Toxicity: {np.mean(toxicity_scores):.4f} (higher = less toxic)")

print("\n--- Configuration Comparison ---")
for config_name, scores in comparison_results.items():
    print(f"  {config_name}:")
    for metric_name, score in scores.items():
        print(f"    {metric_name}: {score:.4f}")

print("\n" + "=" * 70)

---
## 10. Summary & Recommendations

### What We Covered

1. **G-Eval** — Created 4 custom criteria (coherence, completeness, conciseness, helpfulness) and compared their scores.
2. **Custom Metrics** — Built a deterministic metric (response length), a pattern-matching metric (citation format), and an LLM-based metric (domain accuracy).
3. **EvaluationDataset** — Created datasets from scratch, loaded from JSON/CSV, and demonstrated the pytest integration pattern.
4. **Synthetic Data Generation** — Used DeepEval's Synthesizer to auto-generate test cases from documents.
5. **Batch Evaluation** — Compared different pipeline configurations with hyperparameter tracking.
6. **Safety Metrics** — Evaluated outputs for bias and toxicity.

### Recommended Evaluation Stack

| Stage | Metrics | When to Run |
|---|---|---|
| Development | All 5 RAG metrics + G-Eval | Every prompt/config change |
| CI/CD | AnswerRelevancy + Faithfulness | Every PR |
| Pre-release | Full suite + Safety metrics | Before deployment |
| Production | Faithfulness + Safety | Ongoing monitoring |

### Next Steps

- Integrate evaluation into your CI/CD pipeline using the pytest pattern
- Build domain-specific custom metrics for your use case
- Generate synthetic test data periodically as your knowledge base grows
- Track metrics over time to detect quality regressions
- Experiment with different LLM models and compare using the batch evaluation pattern